# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [ ]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [ ]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [ ]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [ ]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [ ]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [ ]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)


In [ ]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.487028,0.381455,0.838235,0.886986
2,0.328676,0.357605,0.848039,0.893471
3,0.167169,0.416541,0.860294,0.901554


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.167169,0.416541,3,0.860294,0.901554


{'eval_loss': 0.4165409803390503,
 'eval_accuracy': 0.8602941176470589,
 'eval_f1': 0.9015544041450777}

## 4. Перевірка на власних прикладах

In [ ]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 86.0%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 2.6%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.0%



## 4 (продовження). Додаткові перевірки на власних даних (завдання 10)

Чотири нові пари: очевидний парафраз, очевидний не-парафраз, пара з однаковими словами, але різним змістом, і пара з запереченням.

In [ ]:
my_pairs = [
    ("The meeting was moved to Friday afternoon.", "They rescheduled the meeting for Friday afternoon."),
    ("The meeting was moved to Friday afternoon.", "The cat slept on the sofa all day."),
    ("The dog bit the man.", "The man bit the dog."),
    ("The new phone is not cheap.", "The new phone is very affordable."),
]
for s1, s2 in my_pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The meeting was moved to Friday afternoon.
They rescheduled the meeting for Friday afternoon.
→ парафраз: 89.7%

The meeting was moved to Friday afternoon.
The cat slept on the sofa all day.
→ парафраз: 3.1%

The dog bit the man.
The man bit the dog.
→ парафраз: 98.1%

The new phone is not cheap.
The new phone is very affordable.
→ парафраз: 97.5%



## 5. Експеримент: більше епох (завдання 9)

Тренуємо нову модель 10 епох (замість 3) і дивимось, як змінюються train loss, validation loss і метрики.

In [ ]:
import pandas as pd

set_seed(42)
model_long = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
args_long = TrainingArguments(
    output_dir="bert-mrpc-long",
    num_train_epochs=10,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)
trainer_long = Trainer(
    model=model_long, args=args_long,
    train_dataset=tokenized_datasets["train"], eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator, compute_metrics=compute_metrics,
)
trainer_long.train()

log = pd.DataFrame(trainer_long.state.log_history)
train_loss = log[log["loss"].notna()][["epoch", "loss"]].rename(columns={"loss": "train_loss"})
evals = log[log["eval_loss"].notna()][["epoch", "eval_loss", "eval_accuracy", "eval_f1"]]
print(train_loss.merge(evals, on="epoch").round(4).to_string(index=False))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.526921,0.398554,0.799020,0.846442
2,0.302837,0.382299,0.845588,0.893039
3,0.160963,0.492709,0.843137,0.882353
4,0.082277,0.609630,0.865196,0.905660
5,0.040089,0.665817,0.867647,0.903571
6,0.018269,0.865115,0.855392,0.898451
7,0.014378,0.954503,0.850490,0.896785
8,0.010017,0.878268,0.855392,0.897747
9,0.006631,0.943451,0.862745,0.904110
10,0.001334,0.931049,0.860294,0.902230


 epoch  train_loss  eval_loss  eval_accuracy  eval_f1
   1.0      0.5269     0.3986         0.7990   0.8464
   2.0      0.3028     0.3823         0.8456   0.8930
   3.0      0.1610     0.4927         0.8431   0.8824
   4.0      0.0823     0.6096         0.8652   0.9057
   5.0      0.0401     0.6658         0.8676   0.9036
   6.0      0.0183     0.8651         0.8554   0.8985
   7.0      0.0144     0.9545         0.8505   0.8968
   8.0      0.0100     0.8783         0.8554   0.8977
   9.0      0.0066     0.9435         0.8627   0.9041
  10.0      0.0013     0.9310         0.8603   0.9022


**Результати 10 епох:**
| епоха | train loss | val loss | accuracy | F1 |
|---|---|---|---|---|
| 1 | 0.5269 | 0.3986 | 0.7990 | 0.8464 |
| 2 | 0.3028 | 0.3823 | 0.8456 | 0.8930 |
| 3 | 0.1610 | 0.4927 | 0.8431 | 0.8824 |
| 4 | 0.0823 | 0.6096 | 0.8652 | 0.9057 |
| 5 | 0.0401 | 0.6658 | 0.8676 | 0.9036 |
| 6 | 0.0183 | 0.8651 | 0.8554 | 0.8985 |
| 7 | 0.0144 | 0.9545 | 0.8505 | 0.8968 |
| 8 | 0.0100 | 0.8783 | 0.8554 | 0.8977 |
| 9 | 0.0066 | 0.9435 | 0.8627 | 0.9041 |
| 10 | 0.0013 | 0.9310 | 0.8603 | 0.9022 |

Для порівняння: тренування на 3 епохи дало accuracy 0.8603 і F1 0.9016.

Train loss падає майже до нуля, а validation loss мінімальний на 2-й епосі (0.382) і далі зростає до ≈ 0.93. Accuracy і F1 при цьому не падають, а коливаються в межах 0.85–0.87 і 0.89–0.91. Тобто модель перенавчається: вона запам'ятовує тренувальні пари й стає надто впевненою в помилкових відповідях (loss росте), хоча частка правильних відповідей майже не змінюється. Нових знань після 2–4 епох вона не здобуває.

## 6. Відповіді на теоретичні питання (0.1, питання 1–10)

**1. Як довго виконувалися клітинки? Чи були проблеми з запуском?**
Усі клітинки виконувалися приблизно від 3 до 8 хвилин. Завантаження датасету, токенізація та тренування моделі на GPU T4 пройшли успішно. Проблем із запуском або помилок не виникало, усе працювало коректно.

**2. Що таке токен `[CLS]` і як він виконує свою функцію?**
`[CLS]` — службовий токен, який додається на початок кожної послідовності. Через механізм self-attention у кожному з 12 шарів він «бачить» усі інші токени, тому його вихідний вектор (768 чисел) стає стислим представленням усього входу (у нашій задачі — усієї пари речень).
Під час попереднього тренування BERT цей вектор використовувався для завдання передбачення наступного речення, тож модель вже вміє складати в нього інформацію про зв'язок між реченнями. Голова-класифікатор бере саме вектор `[CLS]` і перетворює його на 2 числа (логіти) для класів «парафраз / не парафраз».

**3. Яке пряме застосування базової моделі `bert-base-uncased`?**
Без донавчання вона вміє розв'язувати ті задачі, на яких тренувалась: заповнення пропущених слів (masked language modeling, як у `fill-mask` вище) і передбачення наступного речення. Але за карткою моделі її головне призначення — бути основою для донавчання на подальших задачах (класифікація, розмітка послідовностей, відповіді на питання).

**4. Чи варто боротися з проявами сексизму LLM-моделей і як?**
Так. Для `The man worked as a [MASK]` і `The woman worked as a [MASK]` модель пропонує різні набори професій, і для жінок вони помітно більш стереотипні (це видно у виводі вище). Причина: модель вчиться на текстах з інтернету, де ці стереотипи є, і відтворює їх. Якщо така модель використовується в реальних системах (відбір резюме, рекомендації), упередження призводять до реальної дискримінації.
Способи: аудит моделей на спеціальних тестах (WinoBias, StereoSet); аналіз і балансування тренувальних даних (наприклад, контрфактична заміна «він» ↔ «вона»); донавчання й методи дебайасингу; фільтрація або виправлення виходів; прозорість (model cards з описом відомих упереджень). Повністю прибрати упередження неможливо, бо вони є в самих даних, тому їх потрібно вимірювати й обмежувати, а не вважати проблему розв'язаною.

**5. Для чого виводяться числа 0 і 1 у секції «Токенізація пари речень»?**
Це `token_type_ids`: 0 позначає токени першого речення (разом із `[CLS]` і першим `[SEP]`), 1 — токени другого речення (разом з останнім `[SEP]`). Так BERT розрізняє, де закінчується одне речення й починається інше, хоча на вхід вони йдуть однією послідовністю.

**6. Що означають числа 768 і 2?**
768 — розмірність прихованого стану BERT-base (hidden size): стільки чисел у векторі `[CLS]`, що йде на вхід голови. 2 — кількість класів (`num_labels=2`): вихід голови. Тобто голова — це лінійний шар `Linear(768 → 2)`, у ньому 768·2 + 2 = 1538 параметрів.

**7. Те, що в кістяку набагато більше параметрів, ніж у голові, — це добре чи погано?**
Загалом добре. Кістяк (≈109 млн параметрів) уже «розуміє» мову після попереднього тренування, тож голові лишається тільки навчитися зіставляти це розуміння з нашими двома класами; для цього достатньо 3,7 тис. прикладів. Якби тренувати всю мережу з нуля, цих даних було б замало.
Ризики: повне донавчання великого кістяка на малому датасеті може привести до перенавчання і до втрати частини загальних знань, тому використовують малий learning rate (2e-5) і мало епох.

**8. Що вимірюють accuracy і F1?**
Обидві метрики рахуються на валідаційному наборі MRPC (408 пар). Accuracy — частка пар, для яких передбачений клас збігається з еталонним. F1 — гармонійне середнє precision і recall для класу «парафраз» (1): precision показує, яка частка знайдених парафраз справді є парафразами, recall — яку частку справжніх парафраз модель знайшла. F1 потрібна, бо класи незбалансовані (близько двох третин пар — парафрази), і висока accuracy може бути досягнута простим вгадуванням більшого класу.

**9. Збільшіть кількість епох** — див. розділ 5 вище (результати й пояснення).

**10. Додати ще 4 перевірки** — див. розділ 4 (продовження) вище. Дві очевидні пари модель визначила правильно: збіг змісту дав 89.7%, а речення про кота й зустріч — 3.1%. Дві інші пари вона визначила неправильно: «The dog bit the man» / «The man bit the dog» отримали 98.1% парафраза, а «not cheap» / «very affordable» — 97.5%. Обидві пари не є парафразами. Модель орієнтується на лексичний збіг і не помічає зміни порядку слів, що міняє суб'єкт і об'єкт, а також заперечення й антоніми.